# Training a pix2pixHD Global Model using PyTorch / ONNX

This notebook trains a single-stage `pix2pixHD` global generator on paired images.

Dataset format:
- each training file is one `2048x1024` RGB image
- the left half is the target image
- the right half is the input image
- each half becomes a `1024x1024` pair during training

The whole flow is self-contained in this notebook. Run the cells from top to bottom.

In [ ]:
# Make sure you are connected to a runtime with a GPU
!nvidia-smi -L

In [ ]:
# Install required packages
!pip install matplotlib tqdm onnx

In [ ]:
# Import all other dependencies
import copy
import glob
import math
import os
import random
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as TF
from IPython.display import clear_output
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.utils import save_image
from tqdm import tqdm

In [ ]:
# Check if GPU is available
gpu_available = torch.cuda.is_available()
print("GPU is", "available" if gpu_available else "NOT AVAILABLE")

In [ ]:
# OPTIONAL: If you don't have a dataset yet, you can download a pre-existing dataset by executing this cell.
# !curl -o ds.zip https://algorithmicgaze.s3.amazonaws.com/datasets/does-not-exist.zip
# !mkdir -p datasets/faces
# !unzip -j -o -qq ds.zip -d datasets/faces
# !rm -rf datasets/faces/._*

In [ ]:
# Helper functions and training configuration
def directory_should_exist(*parts):
    path = os.path.join(*parts)
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Path '{path}' is not a directory.")
    return path

def ensure_directory(*parts):
    path = os.path.join(*parts)
    os.makedirs(path, exist_ok=True)
    return path

input_dir = directory_should_exist("datasets/faces")
output_dir = ensure_directory("output-pix2pixhd-global")
samples_dir = ensure_directory(output_dir, "samples")

sample_interval = 200
checkpoint_interval = 5
epochs = 100
batch_size = 1
num_workers = 2
train_size = 1024
restart = False
input_on_right = True
use_vgg_loss = True
export_onnx = True

learning_rate = 0.0002
beta1 = 0.5
ngf = 64
ndf = 64
num_D = 2
n_layers_D = 3
n_downsample_global = 4
n_blocks_global = 9
lambda_feat = 10.0

In [ ]:
# Create the dataset class and dataloader
class PairedSplitDataset(Dataset):
    def __init__(self, root_dir, train_size=1024, input_on_right=True, augment=True):
        self.root_dir = root_dir
        self.train_size = train_size
        self.input_on_right = input_on_right
        self.augment = augment
        self.image_files = sorted(
            f
            for f in os.listdir(root_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        )
        if not self.image_files:
            raise RuntimeError(f"No dataset images found in '{root_dir}'.")

    def __len__(self):
        return len(self.image_files)

    def _split_halves(self, image):
        width, height = image.size
        if width != height * 2:
            raise ValueError(
                f"Expected a 2:1 paired image (for example 2048x1024), got {width}x{height}."
            )

        left = image.crop((0, 0, width // 2, height))
        right = image.crop((width // 2, 0, width, height))
        if self.input_on_right:
            return right, left
        return left, right

    def _normalize(self, tensor):
        return (tensor * 2.0) - 1.0

    def _paired_transform(self, input_image, target_image):
        input_image = TF.resize(input_image, [self.train_size, self.train_size], interpolation=TF.InterpolationMode.BICUBIC)
        target_image = TF.resize(target_image, [self.train_size, self.train_size], interpolation=TF.InterpolationMode.BICUBIC)

        if self.augment and random.random() > 0.5:
            input_image = TF.hflip(input_image)
            target_image = TF.hflip(target_image)

        input_tensor = self._normalize(TF.to_tensor(input_image))
        target_tensor = self._normalize(TF.to_tensor(target_image))
        return input_tensor, target_tensor

    def __getitem__(self, index):
        file_name = self.image_files[index]
        image_path = os.path.join(self.root_dir, file_name)
        image = Image.open(image_path).convert("RGB")
        input_image, target_image = self._split_halves(image)
        input_tensor, target_tensor = self._paired_transform(input_image, target_image)
        return input_tensor, target_tensor, file_name

dataset = PairedSplitDataset(
    input_dir,
    train_size=train_size,
    input_on_right=input_on_right,
    augment=True,
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

print(f"Found {len(dataset)} training images.")

In [ ]:
# Show a single image pair from the dataset
def plot_image(subplot, title, tensor):
    image = ((tensor.detach().cpu() + 1.0) / 2.0).clamp(0, 1)
    image = image.permute(1, 2, 0).numpy()
    subplot.imshow(image)
    subplot.set_title(title)
    subplot.axis("off")

sample_input, sample_target, sample_name = next(iter(dataloader))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_image(ax1, f"Input ({sample_name[0]})", sample_input[0])
plot_image(ax2, "Target", sample_target[0])
plt.show()

In [ ]:
# pix2pixHD model definitions
def weights_init(module):
    classname = module.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
        if getattr(module, "bias", None) is not None:
            nn.init.zeros_(module.bias.data)
    elif classname.find("BatchNorm2d") != -1:
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.zeros_(module.bias.data)

def get_norm_layer(norm_type="instance"):
    if norm_type == "batch":
        return lambda channels: nn.BatchNorm2d(channels, affine=True)
    if norm_type == "instance":
        return lambda channels: nn.InstanceNorm2d(channels, affine=False, track_running_stats=False)
    raise NotImplementedError(f"Normalization layer '{norm_type}' is not implemented.")

class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer, padding_type="reflect"):
        super().__init__()
        self.conv_block = self.build_conv_block(dim, norm_layer, padding_type)

    def build_conv_block(self, dim, norm_layer, padding_type):
        block = []
        for layer_index in range(2):
            padding = 0
            if padding_type == "reflect":
                block.append(nn.ReflectionPad2d(1))
            elif padding_type == "replicate":
                block.append(nn.ReplicationPad2d(1))
            elif padding_type == "zero":
                padding = 1
            else:
                raise NotImplementedError(f"Padding type '{padding_type}' is not implemented.")

            block.append(nn.Conv2d(dim, dim, kernel_size=3, padding=padding))
            block.append(norm_layer(dim))
            if layer_index == 0:
                block.append(nn.ReLU(True))
        return nn.Sequential(*block)

    def forward(self, x):
        return x + self.conv_block(x)

class GlobalGenerator(nn.Module):
    def __init__(self, input_nc, output_nc, ngf=64, n_downsampling=4, n_blocks=9, norm_type="instance"):
        super().__init__()
        norm_layer = get_norm_layer(norm_type)
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0),
            norm_layer(ngf),
            nn.ReLU(True),
        ]

        for i in range(n_downsampling):
            mult = 2 ** i
            model += [
                nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1),
                norm_layer(ngf * mult * 2),
                nn.ReLU(True),
            ]

        mult = 2 ** n_downsampling
        for _ in range(n_blocks):
            model.append(ResnetBlock(ngf * mult, norm_layer=norm_layer))

        for i in range(n_downsampling):
            mult = 2 ** (n_downsampling - i)
            model += [
                nn.ConvTranspose2d(
                    ngf * mult,
                    int(ngf * mult / 2),
                    kernel_size=3,
                    stride=2,
                    padding=1,
                    output_padding=1,
                ),
                norm_layer(int(ngf * mult / 2)),
                nn.ReLU(True),
            ]

        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0),
            nn.Tanh(),
        ]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

class NLayerDiscriminator(nn.Module):
    def __init__(self, input_nc, ndf=64, n_layers=3, norm_type="instance", get_interm_feat=True):
        super().__init__()
        self.get_interm_feat = get_interm_feat
        self.n_layers = n_layers
        norm_layer = get_norm_layer(norm_type)
        kw = 4
        padw = int(math.ceil((kw - 1.0) / 2))
        sequence = [[nn.Conv2d(input_nc, ndf, kernel_size=kw, stride=2, padding=padw), nn.LeakyReLU(0.2, True)]]

        nf = ndf
        for _ in range(1, n_layers):
            nf_prev = nf
            nf = min(nf * 2, 512)
            sequence += [[
                nn.Conv2d(nf_prev, nf, kernel_size=kw, stride=2, padding=padw),
                norm_layer(nf),
                nn.LeakyReLU(0.2, True),
            ]]

        nf_prev = nf
        nf = min(nf * 2, 512)
        sequence += [[
            nn.Conv2d(nf_prev, nf, kernel_size=kw, stride=1, padding=padw),
            norm_layer(nf),
            nn.LeakyReLU(0.2, True),
        ]]
        sequence += [[nn.Conv2d(nf, 1, kernel_size=kw, stride=1, padding=padw)]]

        if get_interm_feat:
            for index, layers in enumerate(sequence):
                setattr(self, f"model{index}", nn.Sequential(*layers))
        else:
            stream = []
            for layers in sequence:
                stream += layers
            self.model = nn.Sequential(*stream)

    def forward(self, x):
        if not self.get_interm_feat:
            return [self.model(x)]

        result = [x]
        for index in range(self.n_layers + 2):
            model = getattr(self, f"model{index}")
            result.append(model(result[-1]))
        return result[1:]

class MultiscaleDiscriminator(nn.Module):
    def __init__(self, input_nc, ndf=64, n_layers=3, num_D=2, norm_type="instance", get_interm_feat=True):
        super().__init__()
        self.num_D = num_D
        self.n_layers = n_layers
        self.get_interm_feat = get_interm_feat
        for index in range(num_D):
            netD = NLayerDiscriminator(
                input_nc,
                ndf=ndf,
                n_layers=n_layers,
                norm_type=norm_type,
                get_interm_feat=get_interm_feat,
            )
            if get_interm_feat:
                for layer in range(n_layers + 2):
                    setattr(self, f"scale{index}_layer{layer}", getattr(netD, f"model{layer}"))
            else:
                setattr(self, f"layer{index}", netD.model)

        self.downsample = nn.AvgPool2d(3, stride=2, padding=1, count_include_pad=False)

    def singleD_forward(self, model, x):
        if self.get_interm_feat:
            result = [x]
            for layer in model:
                result.append(layer(result[-1]))
            return result[1:]
        return [model(x)]

    def forward(self, x):
        result = []
        input_downsampled = x
        for scale_offset in range(self.num_D):
            scale_index = self.num_D - 1 - scale_offset
            if self.get_interm_feat:
                model = [getattr(self, f"scale{scale_index}_layer{layer}") for layer in range(self.n_layers + 2)]
            else:
                model = getattr(self, f"layer{scale_index}")
            result.append(self.singleD_forward(model, input_downsampled))
            if scale_offset != self.num_D - 1:
                input_downsampled = self.downsample(input_downsampled)
        return result

In [ ]:
# Loss functions
class GANLoss(nn.Module):
    def __init__(self, use_lsgan=True, target_real_label=1.0, target_fake_label=0.0):
        super().__init__()
        self.register_buffer("real_label", torch.tensor(target_real_label))
        self.register_buffer("fake_label", torch.tensor(target_fake_label))
        self.loss = nn.MSELoss() if use_lsgan else nn.BCEWithLogitsLoss()

    def _target_tensor(self, prediction, target_is_real):
        label = self.real_label if target_is_real else self.fake_label
        return label.expand_as(prediction)

    def forward(self, predictions, target_is_real):
        if isinstance(predictions[0], list):
            total = 0.0
            for scale_predictions in predictions:
                pred = scale_predictions[-1]
                total = total + self.loss(pred, self._target_tensor(pred, target_is_real))
            return total

        pred = predictions[-1]
        return self.loss(pred, self._target_tensor(pred, target_is_real))

def discriminator_feature_matching_loss(pred_fake, pred_real, criterion_feat, lambda_feat):
    feat_weights = 4.0 / (len(pred_fake[0]))
    D_weights = 1.0 / len(pred_fake)
    loss = 0.0
    for scale_predictions_fake, scale_predictions_real in zip(pred_fake, pred_real):
        for fake_feature, real_feature in zip(scale_predictions_fake[:-1], scale_predictions_real[:-1]):
            loss = loss + D_weights * feat_weights * criterion_feat(fake_feature, real_feature.detach()) * lambda_feat
    return loss

class Vgg19(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            weights = models.VGG19_Weights.IMAGENET1K_V1
            vgg_features = models.vgg19(weights=weights).features
        except AttributeError:
            try:
                vgg_features = models.vgg19(pretrained=True).features
            except Exception as exc:
                raise RuntimeError(
                    "Could not load VGG19 weights. Set use_vgg_loss = False if you want to train without perceptual loss."
                ) from exc
        except Exception as exc:
            raise RuntimeError(
                "Could not load VGG19 weights. Set use_vgg_loss = False if you want to train without perceptual loss."
            ) from exc

        self.slice1 = nn.Sequential(*[vgg_features[x] for x in range(2)])
        self.slice2 = nn.Sequential(*[vgg_features[x] for x in range(2, 7)])
        self.slice3 = nn.Sequential(*[vgg_features[x] for x in range(7, 12)])
        self.slice4 = nn.Sequential(*[vgg_features[x] for x in range(12, 21)])
        self.slice5 = nn.Sequential(*[vgg_features[x] for x in range(21, 30)])
        for parameter in self.parameters():
            parameter.requires_grad = False

    def forward(self, x):
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        h5 = self.slice5(h4)
        return [h1, h2, h3, h4, h5]

class VGGLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.vgg = Vgg19().to(device)
        self.criterion = nn.L1Loss()
        self.weights = [1.0 / 32, 1.0 / 16, 1.0 / 8, 1.0 / 4, 1.0]

    def forward(self, x, y):
        x_vgg = self.vgg(x)
        y_vgg = self.vgg(y)
        loss = 0.0
        for weight, feature_x, feature_y in zip(self.weights, x_vgg, y_vgg):
            loss = loss + weight * self.criterion(feature_x, feature_y.detach())
        return loss

In [ ]:
# Checkpoint helpers and a quick smoke test
def create_models(device):
    generator = GlobalGenerator(
        input_nc=3,
        output_nc=3,
        ngf=ngf,
        n_downsampling=n_downsample_global,
        n_blocks=n_blocks_global,
    ).to(device)
    discriminator = MultiscaleDiscriminator(
        input_nc=6,
        ndf=ndf,
        n_layers=n_layers_D,
        num_D=num_D,
        get_interm_feat=True,
    ).to(device)
    generator.apply(weights_init)
    discriminator.apply(weights_init)
    return generator, discriminator

def create_optimizers(generator, discriminator):
    g_optimizer = optim.Adam(generator.parameters(), lr=learning_rate, betas=(beta1, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(beta1, 0.999))
    return g_optimizer, d_optimizer

def get_latest_snapshot(output_dir):
    latest_path = os.path.join(output_dir, "snapshot_latest.pth")
    if os.path.isfile(latest_path):
        return latest_path
    snapshots = glob.glob(os.path.join(output_dir, "snapshot_epoch_*.pth"))
    if not snapshots:
        return None
    return max(snapshots, key=os.path.getctime)

def save_checkpoint(generator, discriminator, g_optimizer, d_optimizer, epoch, output_dir, latest=False):
    file_name = "snapshot_latest.pth" if latest else f"snapshot_epoch_{epoch:04d}.pth"
    checkpoint_path = os.path.join(output_dir, file_name)
    torch.save(
        {
            "epoch": epoch,
            "generator": generator.state_dict(),
            "discriminator": discriminator.state_dict(),
            "g_optimizer": g_optimizer.state_dict(),
            "d_optimizer": d_optimizer.state_dict(),
            "train_size": train_size,
            "model": "pix2pixhd-global",
        },
        checkpoint_path,
    )
    return checkpoint_path

def load_checkpoint(generator, discriminator, g_optimizer, d_optimizer, snapshot_path, device):
    try:
        checkpoint = torch.load(snapshot_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(snapshot_path, map_location=device)

    generator.load_state_dict(checkpoint["generator"])
    discriminator.load_state_dict(checkpoint["discriminator"])
    g_optimizer.load_state_dict(checkpoint["g_optimizer"])
    d_optimizer.load_state_dict(checkpoint["d_optimizer"])
    return int(checkpoint["epoch"]) + 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
smoke_generator, smoke_discriminator = create_models(device)
smoke_g_optimizer, smoke_d_optimizer = create_optimizers(smoke_generator, smoke_discriminator)
criterion_gan = GANLoss().to(device)
criterion_feat = nn.L1Loss()
criterion_vgg = VGGLoss(device) if use_vgg_loss else None

smoke_size = min(train_size, 128)
smoke_input = torch.randn(1, 3, smoke_size, smoke_size, device=device)
smoke_target = torch.randn(1, 3, smoke_size, smoke_size, device=device)
smoke_fake = smoke_generator(smoke_input)

smoke_d_optimizer.zero_grad(set_to_none=True)
smoke_pred_real = smoke_discriminator(torch.cat([smoke_input, smoke_target], dim=1))
smoke_pred_fake = smoke_discriminator(torch.cat([smoke_input, smoke_fake.detach()], dim=1))
smoke_loss_D = 0.5 * (criterion_gan(smoke_pred_real, True) + criterion_gan(smoke_pred_fake, False))
smoke_loss_D.backward()
smoke_d_optimizer.step()

smoke_g_optimizer.zero_grad(set_to_none=True)
smoke_fake = smoke_generator(smoke_input)
smoke_pred_fake = smoke_discriminator(torch.cat([smoke_input, smoke_fake], dim=1))
with torch.no_grad():
    smoke_pred_real = smoke_discriminator(torch.cat([smoke_input, smoke_target], dim=1))
smoke_loss_G = criterion_gan(smoke_pred_fake, True)
smoke_loss_G = smoke_loss_G + discriminator_feature_matching_loss(smoke_pred_fake, smoke_pred_real, criterion_feat, lambda_feat)
if criterion_vgg is not None:
    smoke_loss_G = smoke_loss_G + criterion_vgg(smoke_fake, smoke_target) * lambda_feat
smoke_loss_G.backward()
smoke_g_optimizer.step()
print(f"Smoke test passed at {smoke_size}x{smoke_size}.")

In [ ]:
# Training loop and ONNX export helpers
def save_preview(input_tensor, fake_tensor, target_tensor, output_path):
    triptych = torch.cat([input_tensor.cpu(), fake_tensor.cpu(), target_tensor.cpu()], dim=-1)
    save_image((triptych + 1.0) / 2.0, output_path)

def show_preview(input_tensor, fake_tensor, target_tensor, epoch, step):
    triptych = torch.cat([input_tensor.cpu(), fake_tensor.cpu(), target_tensor.cpu()], dim=-1)
    image = ((triptych.squeeze(0) + 1.0) / 2.0).clamp(0, 1).permute(1, 2, 0).numpy()
    clear_output(wait=True)
    print(f"Epoch {epoch} | Step {step}")
    plt.figure(figsize=(12, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

def export_generator_to_onnx(generator, output_dir, epoch, device):
    onnx_path = os.path.join(output_dir, f"generator_epoch_{epoch:04d}.onnx")
    was_training = generator.training
    generator.eval()
    dummy_input = torch.randn(1, 3, train_size, train_size, device=device)
    with torch.no_grad():
        _ = generator(dummy_input)
    torch.onnx.export(
        generator,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    if was_training:
        generator.train()
    print(f"ONNX model exported to {onnx_path}")

def train(generator, discriminator, dataloader, opts):
    g_optimizer, d_optimizer = create_optimizers(generator, discriminator)
    criterion_gan = GANLoss().to(opts.device)
    criterion_feat = nn.L1Loss()
    criterion_vgg = VGGLoss(opts.device) if opts.use_vgg_loss else None

    fixed_input, fixed_target, _ = next(iter(dataloader))
    fixed_input = fixed_input[:1].to(opts.device)
    fixed_target = fixed_target[:1].to(opts.device)

    start_epoch = 1
    if not opts.restart:
        latest_snapshot = get_latest_snapshot(opts.output_dir)
        if latest_snapshot:
            start_epoch = load_checkpoint(
                generator,
                discriminator,
                g_optimizer,
                d_optimizer,
                latest_snapshot,
                opts.device,
            )
            print(f"Resuming training from epoch {start_epoch - 1}")
        else:
            print("No snapshots found. Starting from scratch.")
    else:
        print("Restarting training from scratch.")

    for epoch in range(start_epoch, opts.epochs + 1):
        last_losses = {}
        progress = tqdm(dataloader, desc=f"Epoch {epoch}/{opts.epochs}")
        for step, (input_img, target_img, file_names) in enumerate(progress, start=1):
            input_img = input_img.to(opts.device, non_blocking=True)
            target_img = target_img.to(opts.device, non_blocking=True)

            d_optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                fake_detached = generator(input_img)
            pred_real = discriminator(torch.cat([input_img, target_img], dim=1))
            pred_fake = discriminator(torch.cat([input_img, fake_detached], dim=1))
            loss_D_real = criterion_gan(pred_real, True)
            loss_D_fake = criterion_gan(pred_fake, False)
            loss_D = 0.5 * (loss_D_real + loss_D_fake)
            loss_D.backward()
            d_optimizer.step()

            g_optimizer.zero_grad(set_to_none=True)
            fake_img = generator(input_img)
            pred_fake = discriminator(torch.cat([input_img, fake_img], dim=1))
            with torch.no_grad():
                pred_real_for_feat = discriminator(torch.cat([input_img, target_img], dim=1))

            loss_G_GAN = criterion_gan(pred_fake, True)
            loss_G_GAN_Feat = discriminator_feature_matching_loss(
                pred_fake,
                pred_real_for_feat,
                criterion_feat,
                opts.lambda_feat,
            )
            loss_G = loss_G_GAN + loss_G_GAN_Feat
            loss_G_VGG = torch.tensor(0.0, device=opts.device)
            if criterion_vgg is not None:
                loss_G_VGG = criterion_vgg(fake_img, target_img) * opts.lambda_feat
                loss_G = loss_G + loss_G_VGG
            loss_G.backward()
            g_optimizer.step()

            last_losses = {
                "loss_D": loss_D.item(),
                "loss_G_GAN": loss_G_GAN.item(),
                "loss_G_GAN_Feat": loss_G_GAN_Feat.item(),
                "loss_G_VGG": loss_G_VGG.item(),
            }
            progress.set_postfix({k: f"{v:.3f}" for k, v in last_losses.items()})

            if step % opts.sample_interval == 0 or step == 1:
                was_training = generator.training
                generator.eval()
                with torch.no_grad():
                    fake_preview = generator(fixed_input)
                preview_path = os.path.join(opts.samples_dir, f"epoch_{epoch:04d}_step_{step:05d}.jpg")
                show_preview(fixed_input, fake_preview, fixed_target, epoch, step)
                save_preview(fixed_input, fake_preview, fixed_target, preview_path)
                if was_training:
                    generator.train()

        save_checkpoint(generator, discriminator, g_optimizer, d_optimizer, epoch, opts.output_dir, latest=True)
        print(f"Epoch {epoch} finished with losses: {last_losses}")

        if epoch % opts.checkpoint_interval == 0 or epoch == opts.epochs:
            checkpoint_path = save_checkpoint(
                generator,
                discriminator,
                g_optimizer,
                d_optimizer,
                epoch,
                opts.output_dir,
                latest=False,
            )
            print(f"Checkpoint saved to {checkpoint_path}")
            if opts.export_onnx:
                export_generator_to_onnx(generator, opts.output_dir, epoch, opts.device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator, discriminator = create_models(device)

opts = SimpleNamespace(
    output_dir=output_dir,
    samples_dir=samples_dir,
    sample_interval=sample_interval,
    checkpoint_interval=checkpoint_interval,
    epochs=epochs,
    restart=restart,
    use_vgg_loss=use_vgg_loss,
    export_onnx=export_onnx,
    lambda_feat=lambda_feat,
    device=device,
)

train(generator, discriminator, dataloader, opts)